# U2_T4 · Cuaderno 1 — Equipo y caso

**Mecatrónica · Métodos Numéricos**

Este cuaderno lo usa **solo el jefe de equipo** y **una sola vez**: identifica al jefe,
**reserva el caso de estudio** y **registra a los integrantes**.

> ⚠️ **Aquí no se hace el ejercicio.** Cuando el equipo quede registrado, pasa al cuaderno
> **`U2_T4_2_Caso`**, que es donde se trabaja.

> ### 👉 Antes de empezar
> - **Solo el jefe** ejecuta este cuaderno.
> - El jefe **descarga el repositorio** y **comparte la carpeta de Drive** con su equipo.
> - El jefe **inicia sesión con su cuenta institucional** (`@campus.tecnm.mx`).

> ✅ **Puedes usar `Ejecutar todo` sin miedo**: ninguna celda reserva ni registra por sí
> sola. El tema y los integrantes se eligen con los botones, y hasta que no los pulses no
> se guarda nada.


### INICIA SESIÓN CON TU CORREO INSTITUCIONAL


In [ ]:
#@title ⚙️ Configuración general — ejecuta esta celda **PRIMERO** (no modificar)
# ============================================================
# Detecta si corre en Google Colab o en tu computadora, monta Drive y
# carga el motor de la tarea (ver MODO_MOTOR más abajo).
# ============================================================
import base64, importlib, os, subprocess, sys, zlib, json

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

# --- Repositorio de la tarea  <-- EDITAR SOLO SI CAMBIA EL REPO ------
REPO_URL = 'https://github.com/ITH-MGL-MN/U2_T4_Casos_de_estudio.git'
REPO_NOMBRE = 'U2_T4_Casos_de_estudio'

# --- Solo en tu computadora: de dónde sale el motor ---
#   "modulos" -> profe/** tal cual, importado como paquete (para depurar)
#   "bundle"  -> grader_local.py, el motor APLANADO (forma exacta de Colab)
#   "blob"    -> grader_ofuscado.txt, lo que recibe el alumno
MODO_MOTOR = "modulos"

if ES_COLAB:
    BASE = "/content/drive/MyDrive/CursoMN"
    REPO = os.path.join(BASE, REPO_NOMBRE)
else:
    REPO = os.getcwd()
    for _ in range(4):
        if os.path.exists(os.path.join(REPO, "grader_local.py")):
            break
        _padre = os.path.dirname(REPO)
        if _padre == REPO:
            break
        REPO = _padre
    if not os.path.exists(os.path.join(REPO, "grader_local.py")):
        REPO = r"o:\Mi unidad\CursoMN\U2_T4_Casos_de_estudio"

MODO = "blob" if ES_COLAB else MODO_MOTOR
print("Modo          :", "Google Colab" if ES_COLAB else "PC local")
print("Carpeta tarea :", REPO)

# ============================================================
# Montar Drive y traer el repositorio  (solo en Colab)
# ============================================================
if ES_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(BASE, exist_ok=True)
    try:
        if not os.path.exists(REPO):
            subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
        else:
            subprocess.run(["git", "-C", REPO, "pull"], check=True)
        print("✅ Repositorio listo en Drive.")
    except Exception as _e:
        print("⚠️ No se pudo clonar/actualizar el repositorio:", _e)
        print("   Si la carpeta ya existe en tu Drive puedes continuar.")
else:
    print("Modo local: no se monta Drive.")

# ============================================================
# Librerías necesarias
# ============================================================
for _pkg in ("numpy", "scipy", "sympy", "matplotlib", "requests", "ipywidgets", "plotly"):
    try:
        importlib.import_module(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

# ============================================================
# Motor de la tarea
# ============================================================
# El motor vive aparte del cuaderno y este recibe solo la API que el motor
# declara en su `__all__`. En Colab se usa el blob; en local, MODO_MOTOR.
sys.path.insert(0, REPO)

if MODO == "modulos":
    from profe.ui.cuaderno import *      # noqa: F401,F403
else:
    if MODO == "blob":
        _ruta = os.path.join(REPO, "grader_ofuscado.txt")
        with open(_ruta, encoding="utf-8") as f:
            _blob = f.read().strip()
        codigo = zlib.decompress(base64.b64decode(_blob)).decode("utf-8")
    else:
        _ruta = os.path.join(REPO, "grader_local.py")
        with open(_ruta, encoding="utf-8") as f:
            codigo = f.read()
    _MOTOR = {}
    try:
        exec(codigo, _MOTOR)
    except Exception as _e:
        raise RuntimeError(
            "No pude cargar el motor desde %s.\n"
            "En local: ejecuta  python tools/build.py  para generarlo.\n"
            "Detalle: %r" % (_ruta, _e))
    globals().update({_n: _MOTOR[_n] for _n in _MOTOR["__all__"]})

print("Motor cargado :", MODO)

# ============================================================
# Preludio estilo MATLAB + robomat
# ============================================================
sys.path.insert(0, os.path.join(REPO, "lib"))
with open(os.path.join(REPO, "lib", "matlab_like.py"), encoding="utf-8") as f:
    exec(f.read(), globals())
from robomat import M, I, sin, cos, tan, var, rad, deg, evaluar, mostrar, jacobiano
print("Preludio MATLAB:", os.path.join(REPO, "lib", "matlab_like.py"))

# ============================================================
# Herramientas de visualización (gráficas interactivas plotly)
# ============================================================
# Viven en tools/visualizar.py, FUERA del motor: no forman parte del sistema
# de calificación, así que puedes leerlas y reutilizarlas.
sys.path.append(os.path.join(REPO, "tools"))
try:
    import visualizar
    importlib.reload(visualizar)
except ImportError as _e:
    raise RuntimeError(
        "No encontré tools/visualizar.py, que las gráficas interactivas usan.\n"
        "En la carpeta de la tarea ese archivo debe estar junto al cuaderno.\n"
        "Detalle: %r" % (_e,))
print("Visualización :", os.path.join(REPO, "tools", "visualizar.py"))

# ============================================================
# Estado del equipo (se guarda en Drive para no perderlo)
# ============================================================
RUTA_ESTADO = os.path.join(REPO, "equipo_u2t4.json")
EQUIPO = None
# El cuaderno 1 guarda aqui el equipo, el correo y el numero de control del jefe;
# asi el cuaderno 2 no tiene que volver a identificarlos.
CORREO = ""
NC_JEFE = ""
_estado = None
if os.path.exists(RUTA_ESTADO):
    try:
        with open(RUTA_ESTADO, encoding="utf-8") as f:
            _estado = json.load(f)
    except Exception as e:
        print("No se pudo leer el estado previo:", e)
if isinstance(_estado, dict) and _estado.get("__estado__") == 1:
    # Formato nuevo (lo escribe este cuaderno): equipo + correo + NC del jefe.
    EQUIPO = _estado.get("equipo")
    CORREO = _estado.get("correo") or ""
    NC_JEFE = _estado.get("nc_jefe") or ""
elif isinstance(_estado, dict):
    # Formato viejo: el equipo a secas. OJO: trae una clave "equipo" con el
    # NUMERO de equipo, por eso la marca de formato es "__estado__" y no "equipo".
    EQUIPO = _estado
if isinstance(EQUIPO, dict):
    # El correo tambien se puede recuperar del integrante jefe (archivos viejos).
    if not CORREO:
        for _i in (EQUIPO.get("integrantes") or []):
            if _i.get("esJefe") and _i.get("correo"):
                CORREO = _i["correo"]
                break
    NC_JEFE = NC_JEFE or EQUIPO.get("jefe", "") or extraer_nc(CORREO) or ""
    print("📄 Estado del equipo recuperado:", EQUIPO.get("tema"), "-", EQUIPO.get("estado"))
else:
    NC_JEFE = NC_JEFE or extraer_nc(CORREO) or ""

import requests

def _post(accion, **campos):
    """Llama a Apps Script y devuelve la carga útil o lanza el error del servidor."""
    payload = {"accion": accion, "token": WEBHOOK_TOKEN}
    payload.update(campos)
    r = requests.post(APPS_SCRIPT_URL, json=payload, timeout=40)
    data = r.json()
    if data.get("status") != "ok":
        raise RuntimeError(data.get("message", "Error desconocido del servidor"))
    return data.get("data", {})

def _guardar_estado():
    # "__estado__" marca el formato: distingue este envoltorio de un archivo
    # viejo, que tambien trae una clave "equipo" (con el numero de equipo).
    with open(RUTA_ESTADO, "w", encoding="utf-8") as f:
        json.dump({"__estado__": 1, "equipo": EQUIPO, "correo": CORREO,
                   "nc_jefe": NC_JEFE}, f, ensure_ascii=False, indent=1)

print("✅ Entorno listo. Ahora ejecuta la celda de identificación.")


## Identifícate (jefe de equipo)

Te identificarás con la cuenta institucional con la que montaste Google Drive.
Después escribe tu **número de control** y se validará contra la lista del grupo.

In [ ]:
# @title Identificación automática del jefe de equipo
import ipywidgets as widgets
from IPython.display import display, HTML

if ES_COLAB:
    from google.colab import auth
    import google.auth, google.auth.transport.requests
    try:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        info = requests.get(
            'https://www.googleapis.com/drive/v3/about?fields=user',
            headers={'Authorization': 'Bearer ' + creds.token}, timeout=20).json()
        CORREO = info.get('user', {}).get('emailAddress', '')
    except Exception as e:
        print('⚠️ No se pudo obtener tu correo automáticamente.', e)
        CORREO = ''
else:
    CORREO = 'm16330887@hermosillo.tecnm.mx'
    # CORREO = input('Correo institucional (pruebas): ').strip()

if CORREO and '@' in CORREO and CORREO.lower().endswith('.tecnm.mx'):
    print('✅ Identificado: ' + CORREO)
elif CORREO:
    display(HTML('<p style="color:#b00020;background:#ffe0e0;padding:12px;border-radius:8px;'
                 'font-weight:bold">⛔ Este correo no es institucional.<br>'
                 'Usa tu cuenta @campus.tecnm.mx.</p>'))
else:
    print('No se pudo identificar el correo. Vuelve a intentar.')

# Número de control sugerido a partir del correo (si trae los 8 dígitos)
NC_JEFE = extraer_nc(CORREO) or (EQUIPO.get('jefe') if EQUIPO else '')
print('NC del jefe:', NC_JEFE)

# Validación contra la lista del grupo
if NC_JEFE:
    _al = _post('proyecto_equipo', NC=NC_JEFE, correo=CORREO)
    if _al.get('equipo'):
        EQUIPO = _al['equipo']
        _guardar_estado()
        print('✅ Este equipo YA tiene el tema %s (%s).' % (EQUIPO['tema'], EQUIPO['estado']))
    else:
        print('✅ NC válido y sin equipo asignado. Sigue con la selección de tema.')
else:
    print('⛔ No pude obtener tu número de control del correo institucional.')

# Guarda el Notebook
Ve a `Archivo → Guardar una copia en Drive` y guárdalo en
`CursoMN/U2_T4_1_Equipo` para conservar tu trabajo.


---
# 1. Elige tu caso de estudio

Esta tarea **no** usa semilla por número de control: **el equipo elige su tema**.

| Paso | Quién | Celda |
|---|---|---|
| 1 | Jefe | **Ver temas disponibles** |
| 2 | Jefe | **Reservar tema** ← *queda bloqueado, ya no se puede cambiar* |
| 3 | Jefe | **Integrantes** |
| 4 | Jefe | **Registrar equipo** |
| 5 | Todo el equipo | Trabajar en el notebook |

> ⚠️ **Una vez reservado el tema, ya no se puede cambiar.** Si te equivocaste, usa la
> celda **Eliminar equipo** (hasta abajo) y vuelve a empezar.

**Bloques según el tamaño del equipo**

| Tamaño | Bloque | Temas |
|---|---|---|
| 2 | A — Básico | T1 – T4 |
| 3 | B — Intermedio | T5 – T9 |
| 4 | C — Estándar mecatrónica | T10 – T15 |
| 5 | D — Avanzado | T16 – T17 |
| 6 | E — Integrador | T18 – T19 |

El **catálogo completo** (con ecuaciones, incertidumbres y desafíos) está en el
`README.md` de la carpeta del proyecto.

In [ ]:
# @title Ver temas disponibles
from IPython.display import display, HTML

_datos = _post('proyecto_temas')
TEMAS = _datos['temas']
LIBRES = [t for t in TEMAS if t['disponible']]

print('Temas LIBRES: %d de %d\n' % (len(LIBRES), len(TEMAS)))
for t in TEMAS:
    if t['disponible']:
        print('  %-4s (%d integrantes) %s' % (t['id'], t['tamano'], t['titulo']))
print()
for t in TEMAS:
    if not t['disponible']:
        print('  %-4s OCUPADO (%s) por NC %s' % (t['id'], t['estado'], t['jefe']))

if EQUIPO:
    print('\n⚠️ Tu equipo ya tiene el tema %s. No necesitas reservar.' % EQUIPO['tema'])

In [ ]:
# @title Reservar tema (BLOQUEA el tema para tu equipo)
SL_TEMA = widgets.Dropdown(
    options=[('%s — %s' % (t['id'], t['titulo']), t['id']) for t in LIBRES],
    description='Tu tema:', layout=widgets.Layout(width='620px'))
BT_RESERVAR = widgets.Button(description='Verificar disponibilidad y reservar',
                             button_style='warning')
OUT_TEMA = widgets.Output()

def _reservar(_):
    global EQUIPO
    with OUT_TEMA:
        OUT_TEMA.clear_output()
        if EQUIPO:
            print('⚠️ Tu equipo ya tiene el tema %s (%s).' % (EQUIPO['tema'], EQUIPO['estado']))
            return
        if not NC_JEFE:
            print('⛔ Primero ejecuta la celda de identificación.')
            return
        try:
            res = _post('proyecto_reservar', NC_Jefe=NC_JEFE, correo=CORREO, tema=SL_TEMA.value)
        except Exception as e:
            print('⛔ No se pudo reservar:', e)
            print('   Vuelve a ejecutar la celda anterior para ver los temas libres.')
            return
        EQUIPO = res['equipo']
        _guardar_estado()
        print('✅', res['mensaje'])
        print('   Integrantes requeridos:', EQUIPO['tamano'])
        print('   Sigue con la celda de INTEGRANTES.')

BT_RESERVAR.on_click(_reservar)
display(widgets.VBox([SL_TEMA, BT_RESERVAR, OUT_TEMA]))

---
# 2. Integrantes del equipo

Aparecerán **tantos selectores como integrantes** tenga tu tema. El **jefe ya viene
seleccionado** (es quien inició sesión). Elige a tus compañeros y luego registra.

> 🔹 En los temas **T2** y **T4** también se permite registrar con **1 solo integrante**
> (el jefe). En ese caso el selector *Nº integrantes* se pone en **1**.


In [ ]:
# @title Integrantes (un selector por integrante)
SEL = {}

if not EQUIPO:
    print('⛔ Primero reserva tu tema (celda anterior).')
else:
    _al = _post('proyecto_alumnos')['alumnos']
    _opciones = [(a['nombre'], a['nc']) for a in _al]
    _opciones.sort()

    N_MAX = int(EQUIPO.get('tamano', 2) or 2)          # máximo del tema
    N_MIN = int(EQUIPO.get('minimo', N_MAX) or N_MAX)  # mínimo permitido (1 si el tema lo admite)
    N_MIN = max(1, min(N_MIN, N_MAX))

    print('Tema %s — %s' % (EQUIPO['tema'], EQUIPO['titulo']))
    if N_MIN < N_MAX:
        print('Este tema se puede registrar con 1 integrante (solo el jefe) o hasta %d.' % N_MAX)
    else:
        print('Este tema requiere %d integrante(s).' % N_MAX)

    W_NINT = widgets.Dropdown(options=list(range(N_MIN, N_MAX + 1)), value=N_MAX,
                              description='Nº integrantes:',
                              layout=widgets.Layout(width='340px'))
    OUT_INT = widgets.Output()

    def _reconstruir_integrantes(_=None):
        """Dibuja el selector del jefe + (n-1) compañeros según W_NINT."""
        global SEL
        _ya = [i['nc'] for i in EQUIPO.get('integrantes', [])]
        n = int(W_NINT.value)
        _nuevo = {}
        for i in range(n):
            if i == 0:
                _nuevo[i] = widgets.Dropdown(options=_opciones, value=NC_JEFE,
                                             description='Jefe:',
                                             layout=widgets.Layout(width='620px'))
            else:
                _def = _ya[i] if len(_ya) > i else _opciones[0][1]
                _nuevo[i] = widgets.Dropdown(options=_opciones, value=_def,
                                             description='Integrante %d:' % (i + 1),
                                             layout=widgets.Layout(width='620px'))
        SEL = _nuevo
        with OUT_INT:
            OUT_INT.clear_output()
            display(widgets.VBox(list(SEL.values())))

    if N_MIN < N_MAX:
        W_NINT.observe(_reconstruir_integrantes, names='value')
        display(W_NINT)
    _reconstruir_integrantes()
    display(OUT_INT)


In [ ]:
# @title Verificar y registrar equipo
BT_REG = widgets.Button(description='Verificar y registrar equipo', button_style='success')
OUT_REG = widgets.Output()

def _registrar(_):
    global EQUIPO
    with OUT_REG:
        OUT_REG.clear_output()
        try:
            _integrantes = [SEL[i].value for i in range(len(SEL))]
        except Exception:
            print('⛔ Primero ejecuta la celda de integrantes.')
            return
        try:
            res = _post('proyecto_registrar', NC_Jefe=NC_JEFE, correo=CORREO,
                        tema=EQUIPO['tema'], integrantes=_integrantes)
        except Exception as e:
            print('⛔ No se pudo registrar:', e)
            return
        EQUIPO = res['equipo']
        _guardar_estado()
        print('✅', res['mensaje'])
        print('\nEquipo registrado:')
        for i in EQUIPO['integrantes']:
            print('   %s  %s%s' % (i['nc'], i['nombre'], '  (jefe)' if i['esJefe'] else ''))
        print('\n🎉 Listo. Ya no necesitas volver a ejecutar estas celdas.')
        print('   Avanza a la sección 4 con TU caso de estudio.')

BT_REG.on_click(_registrar)
display(widgets.VBox([BT_REG, OUT_REG]))

### Eliminar equipo (solo si hubo un error)

Si se equivocaron de tema o de integrantes, pueden **borrar el equipo** y volver a
empezar. Por seguridad se requieren **dos** condiciones:

1. Cambiar `PERMITIR_ELIMINAR = True` aquí abajo (por defecto `False`).
2. Que **tu** número de control **pertenezca** al equipo.

Así nadie puede borrar el equipo de otros.

In [ ]:
PERMITIR_ELIMINAR = False   # <-- cámbialo a True solo para corregir un error
CONFIRMAR = True            # <-- debe quedar en True para borrar

In [ ]:
# @title Eliminar equipo (requiere PERMITIR_ELIMINAR = True)
if not PERMITIR_ELIMINAR:
    print('🔒 La eliminación está desactivada (PERMITIR_ELIMINAR = False).')
elif not EQUIPO:
    print('No hay equipo registrado en este notebook.')
else:
    try:
        res = _post('proyecto_eliminar', NC_Actual=NC_JEFE, tema=EQUIPO['tema'],
                    confirmar=(CONFIRMAR is True))
        print('🗑️', res['mensaje'])
        EQUIPO = None
        _guardar_estado()
        print('Vuelve a la sección 1 para elegir otro tema.')
    except Exception as e:
        print('⛔ No se pudo eliminar:', e)

---
# Siguiente paso

Con el equipo registrado ya puedes continuar:

1. **Comparte la carpeta** `CursoMN/U2_T4_Casos_de_estudio` de Drive con tus compañeros.
2. Cada integrante abre el cuaderno **`U2_T4_2_Caso`** (su propia copia) y hace ahí el
   ejercicio completo.

> 🔹 El **jefe** es quien reserva el tema; **todos** pueden correr la simulación en su copia
> del cuaderno 2.
